# 4 – Modello spaziale: OLS e SEM (Tanzania)

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/4_spatial_model.R`

- OLS: Migrazione ~ Istruzione
- Moran’s I sui residui OLS
- Spatial Error Model (SEM)

In [ ]:
!pip install -q geopandas pyarrow libpysal esda spreg mapclassify

In [ ]:
import geopandas as gpd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from libpysal.weights import Queen, lag_spatial
from esda import Moran
from spreg import OLS, ML_Error
import warnings
warnings.filterwarnings('ignore')
np.random.seed(123)

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## 1) Caricamento dati e statistiche descrittive

In [ ]:
tz = gpd.read_parquet("tanzania.parquet")
print(tz[["AHMIGRWEMP", "EDEDUCWSEH"]].describe())

## 2) Analisi esplorativa

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(tz["EDEDUCWSEH"], tz["AHMIGRWEMP"], edgecolors="k", lw=0.3)
axes[0].set_xlabel("Education")
axes[0].set_ylabel("Migration")
axes[0].set_title("Scatter")

tz.plot(column="EDEDUCWSEH", legend=True, cmap="YlOrRd",
        legend_kwds={"label": "Education"}, ax=axes[1])
axes[1].set_title("Education")
axes[1].set_axis_off()

tz.plot(column="AHMIGRWEMP", legend=True, cmap="YlOrRd",
        legend_kwds={"label": "Migration"}, ax=axes[2])
axes[2].set_title("Migration")
axes[2].set_axis_off()

plt.tight_layout()
plt.show()

## 3) Regressione OLS

In [ ]:
w = Queen.from_dataframe(tz, silence_warnings=True)
w.transform = "r"

y = tz[["AHMIGRWEMP"]].values
x = tz[["EDEDUCWSEH"]].values

ols = OLS(y, x, w=w, spat_diag=True,
          name_y="AHMIGRWEMP", name_x=["EDEDUCWSEH"])
print(ols.summary)

In [ ]:
tz["residual_ols"] = ols.u.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

norm = TwoSlopeNorm(vmin=tz["residual_ols"].min(), vcenter=0,
                    vmax=tz["residual_ols"].max())
sc = axes[0].scatter(tz["EDEDUCWSEH"], tz["AHMIGRWEMP"],
                     c=tz["residual_ols"], cmap="RdBu_r", norm=norm,
                     edgecolors="k", lw=0.5, s=50)
b0, b1 = float(ols.betas[0]), float(ols.betas[1])
xr = np.linspace(tz["EDEDUCWSEH"].min(), tz["EDEDUCWSEH"].max())
axes[0].plot(xr, b0 + b1 * xr, "k-", lw=1)
axes[0].set_xlabel("Education")
axes[0].set_ylabel("Migration")
axes[0].set_title("Linear Regression")
plt.colorbar(sc, ax=axes[0], label="Residual OLS")

tz.plot(column="residual_ols", legend=True, cmap="RdBu_r",
        norm=norm, ax=axes[1])
axes[1].set_title("Residual Map")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## 4) Moran\u2019s I sui residui OLS

In [ ]:
mi_resid = Moran(ols.u.flatten(), w, permutations=999)
print(f"Moran's I (residui): {mi_resid.I:.4f}")
print(f"p-value (sim):       {mi_resid.p_sim:.4f}")
print("\n\u2192 Autocorrelazione spaziale nei residui!" if mi_resid.p_sim < 0.05
      else "\n\u2192 Nessuna autocorrelazione significativa")

## 5) Spatial Error Model (SEM)

$y = X\beta + u, \quad u = \lambda W u + \varepsilon$

In [ ]:
sem = ML_Error(y, x, w=w,
               name_y="AHMIGRWEMP", name_x=["EDEDUCWSEH"])
print(sem.summary)

In [ ]:
print("\n=== Confronto OLS vs SEM ===")
print(f"{'':20s} {'OLS':>10s} {'SEM':>10s}")
print(f"{'Intercept':20s} {float(ols.betas[0]):10.3f} {float(sem.betas[0]):10.3f}")
print(f"{'Education':20s} {float(ols.betas[1]):10.3f} {float(sem.betas[1]):10.3f}")
print(f"{'lambda':20s} {'\u2014':>10s} {float(sem.lam):10.3f}")
print(f"{'AIC':20s} {ols.aic:10.2f} {sem.aic:10.2f}")